# 10 Tomography resolution

Why matrix imaging: LET and attenuation tomography resolve structure too coarsely at BDT depths. Shows ray coverage
against depth, checkerboard recovery at several sizes, and recovery of a BDT low-velocity, low-Q zone. Event
locations are held fixed at their true positions, which is generous to tomography.

Inputs (from 01, 04 and 08): `velocity_model.npz`, `scenarios.npz`, `synthetic_catalogue.csv`, `det_pairs.csv`,
`designs.json`, `stations.csv`, `tt_hypo/`.

In [ ]:
# --- Imports ---
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy.clients.fdsn import Client
from scipy.ndimage import map_coordinates
from scipy.optimize import minimize
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr

rng = np.random.default_rng(2)

In [ ]:
# --- Site (private) ---
site = json.load(open("../data/site.json"))
DRILL = (site["drill_site"]["lon"], site["drill_site"]["lat"])
ELEV = site["elevation_km"]
KY = 111.32
KX = KY * np.cos(np.radians(DRILL[1]))
km = lambda lon, lat: ((np.asarray(lon) - DRILL[0]) * KX, (np.asarray(lat) - DRILL[1]) * KY)

# --- Velocity model and synthetic catalogue ---
m = np.load("../data/velocity_model.npz")
x, y, z = m["x"], m["y"], m["z"]
events = pd.read_csv("../data/synthetic_catalogue.csv", index_col="event")
ev = events[["x", "y", "depth"]].values

In [ ]:
# --- Network and detection (same as 08) ---
GEONET_R = 13.0                        # GeoNet stations within this distance (km)
UPTIME = 0.95                          # best-case pick rate for our stations, assumed
MIN_PICKS = 10                         # stations with fewer P picks left out of the detection fit

# --- Inversion grid ---
CELL = 1.0                             # block size (km)
XMAX, ZMAX = 12.0, 14.0                # +/- 12 km horizontally, 0-14 km deep
STEP = 0.1                             # ray-tracing step (km)

# --- Data errors and damping ---
SIGMA = {"p": 0.02, "s": 0.04}         # pick errors (s), assumed
SIGMA_TSTAR = 0.003                    # t* error (s), assumed
DAMP_V, DAMP_Q = 10.0, 3.0             # damping for velocity and attenuation inversions

# --- Background Q (NZWM 2.2), km below ground ---
Q_Z = np.array([-1, 1, 3, 5, 8, 15]) + ELEV
Q_P = np.array([104, 211, 233, 124, 85, 282])

# --- Tests ---
CHECK_SIZES = [1.0, 2.0, 3.0]          # checkerboard sizes (km)
SCENARIO = "pos00_dome"                # BDT test: conductor at the drill site, conceptual-model dome
V_DROP = 0.05                          # Vp drop inside the BDT, assumed
Q_DROP = 0.5                           # 1/Q higher by 50% inside the BDT (Q lower by a third), assumed

In [ ]:
# --- GeoNet stations near Rotokawa ---
inv = Client("GEONET").get_stations(network="NZ", latitude=DRILL[1], longitude=DRILL[0],
                                    maxradius=0.6, level="station")
gn = pd.DataFrame([(s.code, s.longitude, s.latitude, s.end_date) for nt in inv for s in nt],
                  columns=["code", "lon", "lat", "end"])
gn["x"], gn["y"] = km(gn.lon, gn.lat)
gn = gn[(np.hypot(gn.x, gn.y) <= GEONET_R) & gn.end.isna()]

# --- Plus our Trilliums ---
st02 = pd.read_csv("../data/stations.csv", index_col="station")
trill = json.load(open("../data/designs.json"))["trillium"]
net = pd.concat([pd.DataFrame(dict(code=gn.code, x=gn.x, y=gn.y)),
                 pd.DataFrame(dict(code=[f"T{k:02d}" for k in range(len(trill))],
                                   x=st02.x[trill].values, y=st02.y[trill].values))], ignore_index=True)

# --- Detection model: chance of a pick against magnitude and distance ---
pairs = pd.read_csv("../data/det_pairs.csv")
good = pairs.groupby("code").p.sum()
pairs = pairs[pairs.code.isin(good.index[good >= MIN_PICKS])]


def nll(c, M, logR, d):
    """Negative log-likelihood of the detection model."""
    p = np.clip(np.clip(c[3], 0.01, 1) / (1 + np.exp(-(c[0] + c[1] * M + c[2] * logR))), 1e-9, 1 - 1e-9)
    return -np.sum(d * np.log(p) + (1 - d) * np.log(1 - p))


coef = {ph: minimize(nll, [0, 2, -3, 0.9], args=(pairs.mag.values, np.log10(pairs.R.values), pairs[ph].values),
                     method="Nelder-Mead", options=dict(maxiter=5000)).x for ph in ("p", "s")}

# --- Which stations pick each event (one random draw per event-station pair) ---
R = np.sqrt((ev[:, None, 0] - net.x.values) ** 2 + (ev[:, None, 1] - net.y.values) ** 2 + ev[:, None, 2] ** 2)
M = events.mag.values[:, None]
u = rng.uniform(size=R.shape)
det = {ph: u < UPTIME / (1 + np.exp(-(c[0] + c[1] * M + c[2] * np.log10(R)))) for ph, c in coef.items()}
print(len(net), "stations;", {ph: int(d.sum()) for ph, d in det.items()}, "picks")

In [ ]:
# --- Blocks ---
ex = np.arange(-XMAX, XMAX + 1e-9, CELL)
ez = np.arange(0, ZMAX + 1e-9, CELL)
nx, nz = len(ex) - 1, len(ez) - 1
cx, cz = ex[:-1] + CELL / 2, ez[:-1] + CELL / 2
CX, CY, CZ = np.meshgrid(cx, cx, cz, indexing="ij")
centre = np.hypot(CX, CY) <= 6                       # blocks within 6 km of the drill site, used for scoring
j0 = np.argmin(abs(cx))                              # block column through the drill site

# --- Background slowness and 1/Q in each block ---
ix_ = [np.argmin(abs(x - c)) for c in cx]
iz_ = [np.argmin(abs(z - c)) for c in cz]
s0 = {ph: (1 / m[f"v{ph}"][np.ix_(ix_, ix_, iz_)]).ravel() for ph in ("p", "s")}
q0 = 1 / np.interp(CZ, Q_Z, Q_P).ravel()


def block_of(p):
    """Index of the block containing each point (-1 if outside)."""
    i = np.floor((p[:, 0] + XMAX) / CELL).astype(int)
    j = np.floor((p[:, 1] + XMAX) / CELL).astype(int)
    k = np.floor(p[:, 2] / CELL).astype(int)
    ok = (i >= 0) & (i < nx) & (j >= 0) & (j < nx) & (k >= 0) & (k < nz)
    return np.where(ok, (i * nx + j) * nz + k, -1)

In [ ]:
dg = np.array([x[1] - x[0], y[1] - y[0], z[1] - z[0]])
g0 = np.array([x[0], y[0], z[0]])


def trace(ph):
    """Ray-length matrix L (picks x blocks) and the (event, station) of each row, for one phase."""
    rows, cols, picks = [], [], []
    for s, st in net.iterrows():
        evs = np.where(det[ph][:, s])[0]
        if len(evs) == 0:
            continue
        T = np.load(f"../data/tt_hypo/{st.code}_{ph}.npy").astype(np.float32)
        grad = np.gradient(T, *dg)
        p = ev[evs].astype(float)
        alive = np.ones(len(evs), bool)
        for _ in range(int(60 / STEP)):
            idx = ((p[alive] - g0) / dg).T
            gv = np.column_stack([map_coordinates(gc, idx, order=1) for gc in grad])
            step = -STEP * gv / np.linalg.norm(gv, axis=1, keepdims=True)       # downhill, towards the station
            b = block_of(p[alive] + step / 2)
            n = np.where(alive)[0]
            rows.append(len(picks) + n[b >= 0])
            cols.append(b[b >= 0])
            p[alive] += step
            alive[alive] = np.hypot(p[alive, 0] - st.x, p[alive, 1] - st.y) + p[alive, 2] > 0.3   # stop at station
            if not alive.any():
                break
        picks += [(e, s) for e in evs]
    r, c = np.concatenate(rows), np.concatenate(cols)
    return coo_matrix((np.full(len(r), STEP), (r, c)), shape=(len(picks), nx * nx * nz)).tocsr()


L = {ph: trace(ph) for ph in ("p", "s")}
print({ph: Lp.shape[0] for ph, Lp in L.items()}, "rays")

In [ ]:
hits = np.asarray((L["p"] > 0).sum(axis=0)).ravel().reshape(nx, nx, nz)

with plt.style.context("default"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
    ax1.barh(cz, [hits[:, :, k][centre[:, :, k]].mean() for k in range(nz)], height=CELL * 0.9)
    ax1.set(xlabel="P rays per block (within 6 km)", ylabel="Depth (km)", ylim=(ZMAX, 0), title="Ray coverage")
    ax2.hist(events.depth, bins=ez, orientation="horizontal", color="grey")
    ax2.set(xlabel="Number of events", title="Event depths")
    for ax in (ax1, ax2):
        ax.axhspan(3.5, 7, color="firebrick", alpha=0.1, label="BDT range")
    ax1.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
Gv = L["p"].multiply(-s0["p"][None, :]).tocsr()              # dt  = Gv @ (dv/v)
Gq = L["p"].multiply((s0["p"] * q0)[None, :]).tocsr()        # dt* = Gq @ (d(1/Q) / (1/Q))


def invert(G, m_true, sigma, damp):
    """Synthetic data with noise, then damped least squares."""
    d = G @ m_true + rng.normal(0, sigma, G.shape[0])
    return lsqr(G / sigma, d / sigma, damp=damp)[0]


def by_depth(m_true, m_est):
    """Correlation between true and recovered patterns at each depth (within 6 km of the drill site)."""
    t, e = m_true.reshape(nx, nx, nz), m_est.reshape(nx, nx, nz)
    return np.array([np.corrcoef(t[:, :, k][centre[:, :, k]], e[:, :, k][centre[:, :, k]])[0, 1]
                     for k in range(nz)])

In [ ]:
check = {}
for size in CHECK_SIZES:
    sign = (-1) ** (np.floor((CX + XMAX) / size) + np.floor((CY + XMAX) / size) + np.floor(CZ / size))
    mv, mq = (0.05 * sign).ravel(), (0.25 * sign).ravel()       # +/- 5% velocity, +/- 25% in 1/Q
    check[size] = (by_depth(mv, invert(Gv, mv, SIGMA["p"], DAMP_V)),
                   by_depth(mq, invert(Gq, mq, SIGMA_TSTAR, DAMP_Q)))

with plt.style.context("default"):
    fig, axs = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
    for size, (cv, cq) in check.items():
        axs[0].plot(cv, cz, "o-", label=f"{size:g} km")
        axs[1].plot(cq, cz, "o-", label=f"{size:g} km")
    for ax, title in zip(axs, ["LET (Vp)", "Attenuation (Qp)"]):
        ax.axhspan(3.5, 7, color="firebrick", alpha=0.1)
        ax.axvline(0.5, color="grey", ls=":")
        ax.set(xlim=(-0.2, 1), ylim=(ZMAX, 0), xlabel="Recovery (correlation)", title=title)
    axs[0].set_ylabel("Depth (km)")
    axs[0].legend(title="Checker size", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
sc = np.load("../data/scenarios.npz")
top, base = sc[f"top_{SCENARIO}"], sc[f"base_{SCENARIO}"]

# --- Fraction of each block inside the BDT zone (sampled every 0.1 km in depth) ---
it = [np.argmin(abs(sc["x"] - c)) for c in cx]
zs = np.arange(0, ZMAX, 0.1)
t3 = np.nan_to_num(top[np.ix_(it, it)], nan=99)[..., None]
b3 = np.nan_to_num(base[np.ix_(it, it)], nan=99)[..., None]
inside = ((zs >= t3) & (zs < b3)).reshape(nx, nx, nz, -1).mean(axis=3).ravel()

# --- Invert ---
mv_true, mq_true = -V_DROP * inside, Q_DROP * inside
mv_est = invert(Gv, mv_true, SIGMA["p"], DAMP_V)
mq_est = invert(Gq, mq_true, SIGMA_TSTAR, DAMP_Q)

# --- Where does each method put the BDT top under the crest? (half the column's peak) ---
for label, mt, me in [("LET", -mv_true, -mv_est), ("Attenuation", mq_true, mq_est)]:
    col, col_t = me.reshape(nx, nx, nz)[j0, j0], mt.reshape(nx, nx, nz)[j0, j0]
    k = np.argmax(col >= 0.5 * col.max())
    print(f"{label}: recovered top ~{cz[k]:.1f} km (true {np.nanmin(top):.1f}), "
          f"peak amplitude {col.max() / col_t.max():.0%} of true")

In [ ]:
panels = [("True BDT zone (dVp/Vp)", -mv_true, V_DROP),
          ("LET recovery (dVp/Vp)", -mv_est, V_DROP),
          ("Attenuation recovery (d(1/Q))", mq_est, Q_DROP)]
ts = top[np.ix_(it, [np.argmin(abs(sc["y"]))])].ravel()

with plt.style.context("default"):
    fig, axs = plt.subplots(1, len(panels), figsize=(11, 3.6), sharey=True, constrained_layout=True)
    for ax, (title, mm, vmax) in zip(axs, panels):
        pc = ax.pcolormesh(ex, ez, mm.reshape(nx, nx, nz)[:, j0, :].T, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.plot(cx, ts, "k--", lw=1)
        ax.set(title=title, xlim=(-8, 8), ylim=(ZMAX, 0), xlabel="East (km)")
        fig.colorbar(pc, ax=ax, shrink=0.8)
    axs[0].set_ylabel("Depth (km)")
    plt.show()